In [3]:
import json
import boto3
import pandas as pd


In [4]:
bucket = "data605-final-project"
prefix = "Talent/"

s3 = boto3.client("s3")

def get_json_files(bucket, prefix=""):
    paginator = s3.get_paginator("list_objects_v2")
    records = {}

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.lower().endswith(".json"):
                continue
            body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
            records[key] = json.loads(body)

    return records

In [5]:
data = get_json_files(bucket, prefix)
print(f"Loaded {len(data)} JSON files")

Loaded 3105 JSON files


In [6]:
tech_key = "tech_self_score"
strengths_key = "strengths"
weaknesses_key = "weaknesses"
not_to_include = {tech_key,strengths_key,weaknesses_key}

candidate_rows, tech_rows, strength_rows, weakness_rows = [], [], [], []
for key, rec in data.items():
    if not isinstance(rec, dict):
        continue

    file_id = key.split("/")[-1].replace(".json", "")
    name = rec.get("name")
    date = rec.get("date")
    
    candidate = {k: v for k, v in rec.items() if k not in not_to_include}
    candidate["source_file"] = key
    candidate["file_id"] = file_id
    candidate_rows.append(candidate)

    for skill, score in (rec.get(tech_key) or {}).items():
        tech_rows.append({"file_id": file_id, "name": name, "date": date, "skill": skill, "score": score})

    for item in (rec.get(strengths_key) or []):
        strength_rows.append({"file_id": file_id, "name": name, "date": date, "strength": item})

    for item in (rec.get(weaknesses_key) or []):
        weakness_rows.append({"file_id": file_id, "name": name, "date": date, "weakness": item})

df_candidate = pd.DataFrame(candidate_rows)
df_tech = pd.DataFrame(tech_rows)
df_strengths = pd.DataFrame(strength_rows)
df_weaknesses = pd.DataFrame(weakness_rows)

print("candidate", df_candidate.shape)
print("tech", df_tech.shape)
print("strengths", df_strengths.shape)
print("weaknesses", df_weaknesses.shape)

print(df_candidate.head())

print(df_tech.head())
print(df_tech.shape)
print(df_tech.dtypes)
print(df_tech.isna().sum())

print(df_strengths.head())
print(df_strengths.shape)
print(df_strengths.dtypes)
print(df_strengths.isna().sum())

print(df_weaknesses.head())
print(df_weaknesses.shape)
print(df_weaknesses.dtypes)
print(df_weaknesses.isna().sum())

candidate (3105, 9)
tech (9770, 5)
strengths (6253, 4)
weaknesses (6164, 4)
                name        date self_development geo_flex  \
0  Stillmann Castano  22/08/2019              Yes      Yes   
1    Hilary Willmore  01/08/2019               No      Yes   
2      Efrem Whipple  22/08/2019              Yes      Yes   
3        Sydel Fenne  28/08/2019              Yes      Yes   
4    Michel Lebarree  07/08/2019              Yes      Yes   

  financial_support_self result course_interest        source_file file_id  
0                    Yes   Pass        Business  Talent/10383.json   10383  
1                    Yes   Fail            Data  Talent/10384.json   10384  
2                    Yes   Pass        Business  Talent/10385.json   10385  
3                    Yes   Pass            Data  Talent/10386.json   10386  
4                    Yes   Pass     Engineering  Talent/10387.json   10387  
  file_id               name        date       skill  score
0   10383  Stillmann Castano 

In [7]:
print("Records:", len(df_candidate))
print("Unique file_ids:", df_candidate["file_id"].nunique())
print("Unique name/date combos:", df_candidate[["name", "date"]].drop_duplicates().shape[0])

Records: 3105
Unique file_ids: 3105
Unique name/date combos: 3073


In [9]:
dupes = df[df.duplicated(subset=["name", "date"], keep=False)]
dupe_ids = dupes["file_id"]
tech_dupes = df_tech[df_tech["file_id"].isin(dupe_ids)]
print(tech_dupes.sort_values(["name", "date", "skill"])[["file_id", "name", "date", "skill", "score"]].to_string())

NameError: name 'df' is not defined

In [10]:
print(df_tech.duplicated(subset=["file_id", "skill"]).sum())
print(len(df_tech))

print("---------------------")
df_tech = df_tech.drop_duplicates(subset=["file_id", "skill"], keep="first")

print(df_tech.duplicated(subset=["file_id", "skill"]).sum())
print(len(df_tech))

print("=====================")

print(df_strengths.duplicated(subset=["file_id", "strength"]).sum())
print(len(df_strengths))

print("---------------------")
df_strengths = df_strengths.drop_duplicates(subset=["file_id", "strength"], keep="first")

print(df_strengths.duplicated(subset=["file_id", "strength"]).sum())
print(len(df_strengths))

print("=====================")

print(df_weaknesses.duplicated(subset=["file_id", "weakness"]).sum())
print(len(df_weaknesses))

print("---------------------")
df_weaknesses = df_weaknesses.drop_duplicates(subset=["file_id", "weakness"], keep="first")

print(df_weaknesses.duplicated(subset=["file_id", "weakness"]).sum())
print(len(df_weaknesses))

0
9770
---------------------
0
9770
0
6253
---------------------
0
6253
0
6164
---------------------
0
6164


In [11]:

print(df_tech.shape)
print(df_tech.dtypes)
print(df_tech.isna().sum())

print("---------------------")

print(df_strengths.shape)
print(df_strengths.dtypes)
print(df_strengths.isna().sum())

print("---------------------")

print(df_weaknesses.shape)
print(df_weaknesses.dtypes)
print(df_weaknesses.isna().sum())

(9770, 5)
file_id    object
name       object
date       object
skill      object
score       int64
dtype: object
file_id    0
name       0
date       0
skill      0
score      0
dtype: int64
---------------------
(6253, 4)
file_id     object
name        object
date        object
strength    object
dtype: object
file_id     0
name        0
date        0
strength    0
dtype: int64
---------------------
(6164, 4)
file_id     object
name        object
date        object
weakness    object
dtype: object
file_id     0
name        0
date        0
weakness    0
dtype: int64


In [12]:
print(df_tech.nunique())
print("=====================")
print(df_strengths.nunique())
print("=====================")
print(df_weaknesses.nunique())

file_id    3050
name       3019
date        171
skill         9
score        12
dtype: int64
file_id     3105
name        3073
date         172
strength      25
dtype: int64
file_id     3105
name        3073
date         172
weakness      24
dtype: int64


In [13]:
print(sorted(df_tech["skill"].unique()))
print("=====================")
print(sorted(df_tech["score"].unique()))
print("=====================")
print(sorted(df_strengths["strength"].unique()))
print("=====================")
print(sorted(df_weaknesses["weakness"].unique()))

['C#', 'C++', 'Java', 'JavaScript', 'PHP', 'Python', 'R', 'Ruby', 'SPSS']
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
['Altruism', 'Ambitious', 'Charisma', 'Collaboration', 'Competitive', 'Composure', 'Consistent', 'Courteous', 'Creative', 'Critical Thinking', 'Curious', 'Determined', 'Efficient', 'Empathy', 'Independent', 'Innovative', 'Listening', 'Organisation', 'Passionate', 'Patient', 'Perfectionism', 'Problem Solving', 'Rational', 'Reliable', 'Versatile']
['Anxious', 'Chaotic', 'Chatty', 'Competitive', 'Controlling', 'Conventional', 'Critical', 'Distracted', 'Immature', 'Impatient', 'Impulsive', 'Indecisive', 'Indifferent', 'Intolerant', 'Introverted', 'Overbearing', 'Passive', 'Perfectionist', 'Procrastination', 'Selfish', 'Sensitive', 'Slow', 'Stubborn', 'Undisciplined']


In [20]:
dates = df_strengths[["file_id", "date"]]
pattern = dates["date"].astype(str).str.replace(r"\d", "9", regex=True).str.replace(r"[A-Za-z]", "a", regex=True)
print(pattern.value_counts())

date
99/99/9999    6253
Name: count, dtype: int64


In [15]:
for frame in [df_tech, df_strengths, df_weaknesses]:
    print((frame["date"].astype(str).str.contains("//")).sum())
    frame["date"] = frame["date"].astype(str).str.replace(r"/{2,}", "/", regex=True)
    print((frame["date"].str.contains("//")).sum())

85
0
54
0
59
0


In [22]:
parts = dates["date"].astype(str).str.extract(r"^(\d{1,2})[/\-.](\d{1,2})[/\-.]")
print((parts[0].astype(float) > 12).sum())
print((parts[1].astype(float) > 12).sum())

3719
0


In [23]:
parts = dates["date"].astype(str).str.extract(r"^(\d{1,2})[/\-.](\d{1,2})[/\-.]\d{2,4}")
first = pd.to_numeric(parts[0])
second = pd.to_numeric(parts[1])

print((first > 12).sum())
print((second > 12).sum())
print(parts[0].isna().sum())

3719
0
0


In [24]:
unmatched = dates[parts[0].isna()]
print(len(unmatched))
print(unmatched.to_string())

0
Empty DataFrame
Columns: [file_id, date]
Index: []


In [25]:
for frame in [df_tech, df_strengths, df_weaknesses]:
    frame["date"] = pd.to_datetime(frame["date"], dayfirst=True, errors="coerce", format="mixed")

for frame in [df_tech, df_strengths, df_weaknesses]:
    print(frame["date"].isna().sum())
    print(frame["date"].dtype)

0
datetime64[ns]
0
datetime64[ns]
0
datetime64[ns]


In [26]:
for frame in [df_tech, df_strengths, df_weaknesses]:
    obj_cols = frame.select_dtypes(include="object").columns
    frame[obj_cols] = frame[obj_cols].astype("string")

In [27]:

print(df_tech.dtypes)

print("---------------------")

print(df_strengths.dtypes)

print("---------------------")

print(df_weaknesses.dtypes)

file_id    string[python]
name       string[python]
date       datetime64[ns]
skill      string[python]
score               int64
dtype: object
---------------------
file_id     string[python]
name        string[python]
date        datetime64[ns]
strength    string[python]
dtype: object
---------------------
file_id     string[python]
name        string[python]
date        datetime64[ns]
weakness    string[python]
dtype: object
